# Dataset D pancreas validation: CellRank VelocityKernel fates

This notebook reconstructs a CellRank `VelocityKernel` from the scVelo dynamical RNA velocity outputs, runs GPCCA, identifies initial and terminal states, computes fate probabilities, and records the CellRank comparator outputs.

The VelocityKernel and the official `preprocessed-kernel` reference are explicitly treated as RNA-velocity-derived evidence, not evidence independent of scVelo.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "scripts"))
from pancreas_validation_common import (
    configured_paths,
    ensure_output_tree,
    ensure_runtime_env,
    load_config,
    rel_display,
    sha256_file,
    version_record,
    write_alt_text,
    write_dataframe,
    write_json,
    write_metadata,
)

CONFIG = load_config(ROOT)
PATHS = configured_paths(CONFIG, ROOT)
DATA_DIR = PATHS["data_dir"]
OUTPUT_DIR = PATHS["output_dir"]
DATA_DIR.mkdir(parents=True, exist_ok=True)
ensure_runtime_env(OUTPUT_DIR)
ensure_output_tree(OUTPUT_DIR)

In [ ]:

import json

import anndata as ad
import cellrank as cr
import numpy as np
import pandas as pd
from scipy import sparse

scvelo_summary = pd.read_csv(OUTPUT_DIR / "figure_sources" / "01_scvelo_summary.csv").iloc[0]
scvelo_path = ROOT / scvelo_summary["output_h5ad"]
if sha256_file(scvelo_path) != scvelo_summary["output_sha256"]:
    raise AssertionError("scVelo h5ad checksum changed after notebook 01")
adata = ad.read_h5ad(scvelo_path)

vk = cr.kernels.VelocityKernel(adata, xkey="Ms", vkey="velocity")
vk.compute_transition_matrix(
    model=CONFIG["cellrank"]["kernel_model"],
    similarity=CONFIG["cellrank"]["kernel_similarity"],
    seed=CONFIG["cellrank"]["seed"],
)
vk.write_to_adata(key=CONFIG["cellrank"]["transition_matrix_key"])

g = cr.estimators.GPCCA(vk)
g.compute_schur(n_components=CONFIG["cellrank"]["n_schur_components"])
g.compute_macrostates(n_states=CONFIG["cellrank"]["n_macrostates"], cluster_key=CONFIG["cluster_key"])
g.predict_terminal_states(method="top_n", n_states=CONFIG["cellrank"]["n_terminal_states"])
g.predict_initial_states(n_states=CONFIG["cellrank"]["n_initial_states"])
g.compute_fate_probabilities(
    solver=CONFIG["cellrank"]["fate_probability_solver"],
    use_petsc=CONFIG["cellrank"]["use_petsc"],
    n_jobs=CONFIG["cellrank"]["n_jobs"],
    show_progress_bar=True,
)

cellrank_h5ad = OUTPUT_DIR / "intermediates" / "pancreas_cellrank_velocitykernel_gpcca.h5ad"
cellrank_h5ad.parent.mkdir(parents=True, exist_ok=True)
try:
    g_adata = g.to_adata(keep="all", copy=True)
    g_adata.write_h5ad(cellrank_h5ad, compression="gzip")
except Exception:
    adata.write_h5ad(cellrank_h5ad, compression="gzip")

estimator_path = OUTPUT_DIR / "intermediates" / "pancreas_gpcca_estimator.pkl"
try:
    g.write(estimator_path, write_adata=False)
    estimator_sha = sha256_file(estimator_path)
except Exception as exc:
    estimator_path = None
    estimator_sha = None
    write_json(OUTPUT_DIR / "metadata" / "02_gpcca_pickle_warning.json", {"warning": repr(exc)})

transition = adata.obsp[CONFIG["cellrank"]["transition_matrix_key"]]
transition_summary = pd.DataFrame([{
    "transition_matrix_key": CONFIG["cellrank"]["transition_matrix_key"],
    "n_rows": int(transition.shape[0]),
    "n_cols": int(transition.shape[1]),
    "nnz": int(transition.nnz) if sparse.issparse(transition) else int(np.count_nonzero(transition)),
    "cellrank_velocity_kernel_independent_of_scvelo": False,
    "dependency_statement": "VelocityKernel transition matrix is computed from scVelo dynamical RNA velocity and is not independent validation evidence.",
}])

def lineage_to_frame(lineage, index):
    if lineage is None:
        return pd.DataFrame(index=index)
    values = np.asarray(lineage)
    names = list(getattr(lineage, "names", [f"lineage_{i}" for i in range(values.shape[1])]))
    return pd.DataFrame(values, index=index, columns=[str(name) for name in names])

fate = lineage_to_frame(g.fate_probabilities, adata.obs_names)
fate.insert(0, "cell_id", adata.obs_names)
fate.insert(1, CONFIG["cluster_key"], adata.obs[CONFIG["cluster_key"]].astype(str).values)
cluster_fate = fate.groupby(CONFIG["cluster_key"], observed=False).mean(numeric_only=True).reset_index()

state_rows = []
for label, series in [("terminal_states", g.terminal_states), ("initial_states", g.initial_states), ("macrostates", g.macrostates)]:
    if series is None:
        continue
    s = pd.Series(series, index=adata.obs_names).astype(str)
    tmp = pd.DataFrame({"cell_id": adata.obs_names, CONFIG["cluster_key"]: adata.obs[CONFIG["cluster_key"]].astype(str).values, label: s.values})
    counts = tmp.groupby([CONFIG["cluster_key"], label], observed=False).size().reset_index(name="n_cells")
    counts.insert(0, "state_type", label)
    state_rows.append(counts.rename(columns={label: "cellrank_state"}))
state_assignments = pd.concat(state_rows, ignore_index=True) if state_rows else pd.DataFrame(columns=["state_type", CONFIG["cluster_key"], "cellrank_state", "n_cells"])

checksums = pd.read_csv(OUTPUT_DIR / "figure_sources" / "00_pancreas_dataset_checksums.csv")
ref_row = checksums.loc[checksums["kind"].eq("preprocessed-kernel")].iloc[0]
ref_path = ROOT / ref_row["path"]
ref_status = "available" if ref_path.exists() and sha256_file(ref_path) == ref_row["sha256"] else "checksum_mismatch_or_missing"
reference_comparator = pd.DataFrame([{
    "comparator": "official_cellrank_preprocessed_kernel",
    "path": rel_display(ref_path, ROOT),
    "checksum_status": ref_status,
    "role": CONFIG["dataset"]["preprocessed_kernel_role"],
    "independent_of_scvelo": False,
}])
if ref_status == "available":
    ref = ad.read_h5ad(ref_path)
    if CONFIG["cellrank"]["transition_matrix_key"] in ref.obsp:
        reference_comparator.loc[0, "transition_matrix_present"] = True
        reference_comparator.loc[0, "n_cells"] = int(ref.n_obs)
        reference_comparator.loc[0, "n_vars"] = int(ref.n_vars)
    else:
        reference_comparator.loc[0, "transition_matrix_present"] = False

write_dataframe(OUTPUT_DIR, "02_cellrank_transition_summary", transition_summary)
write_dataframe(OUTPUT_DIR, "02_cellrank_fate_probabilities", fate)
write_dataframe(OUTPUT_DIR, "02_cellrank_cluster_fate_summary", cluster_fate)
write_dataframe(OUTPUT_DIR, "02_cellrank_state_assignments", state_assignments)
write_dataframe(OUTPUT_DIR, "02_cellrank_reference_comparator", reference_comparator)
write_alt_text(
    OUTPUT_DIR,
    "02_cellrank_fates",
    "CellRank GPCCA was run on a VelocityKernel computed from scVelo dynamical RNA velocity. Terminal states and fate probabilities are complementary velocity-derived comparator outputs, not evidence independent of RNA velocity."
)
write_metadata(OUTPUT_DIR, "02_cellrank_fates", CONFIG, {
    "input_h5ad": rel_display(scvelo_path, ROOT),
    "input_sha256": scvelo_summary["output_sha256"],
    "output_h5ad": rel_display(cellrank_h5ad, ROOT),
    "output_sha256": sha256_file(cellrank_h5ad),
    "gpcca_estimator": rel_display(estimator_path, ROOT) if estimator_path else None,
    "gpcca_estimator_sha256": estimator_sha,
    "cellrank_parameters": CONFIG["cellrank"],
    "cellrank_velocity_kernel_independent_of_scvelo": False,
})
version_record(OUTPUT_DIR, "02_cellrank_fates", CONFIG, {"output_h5ad_sha256": sha256_file(cellrank_h5ad)})
transition_summary
